In [1]:
# RAGAS needs openai key set in environment (which we already did)
import os
from datasets import Dataset

def evaluate_with_ragas(query: str, answer: str, contexts: list, ground_truth: str = None):
    """Evaluate using RAGAS framework."""
    try:
        from ragas import evaluate
        from ragas.metrics import faithfulness, answer_relevancy, context_precision

        data = {
            "question": [query],
            "answer": [answer],
            "contexts": [contexts]
        }
        metrics = [faithfulness, answer_relevancy, context_precision]

        if ground_truth:
            from ragas.metrics import context_recall
            data["ground_truth"] = [ground_truth]
            metrics.append(context_recall)

        dataset = Dataset.from_dict(data)
        results = evaluate(dataset, metrics=metrics)
        scores = results.to_pandas().iloc[0].to_dict()

        return {
            "faithfulness": float(scores.get("faithfulness", 0)),
            "answer_relevancy": float(scores.get("answer_relevancy", 0)),
            "context_precision": float(scores.get("context_precision", 0)),
            "context_recall": float(scores.get("context_recall", 0)) if ground_truth else None
        }
    except Exception as e:
        print(f"⚠️ RAGAS failed ({e}), using heuristic fallback")
        return heuristic_eval(query, answer, contexts)

def heuristic_eval(query, answer, contexts):
    """Fast heuristic when RAGAS fails."""
    answer_words = set(answer.lower().split())
    ctx_words = set(" ".join(contexts).lower().split())
    query_words = set(query.lower().split())

    faithfulness = min(len(answer_words & ctx_words) / max(len(answer_words), 1), 1.0)
    relevancy = min(len(query_words & answer_words) / max(len(query_words), 1) * 2, 1.0)
    return {"faithfulness": faithfulness, "answer_relevancy": relevancy, "context_precision": 0.5}

print("✅ Evaluation functions ready")

✅ Evaluation functions ready


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
def full_rag_pipeline(query: str, verbose: bool = False):
    """Placeholder for the full RAG pipeline.

    This function should implement your RAG logic, including:
    1. Optional query rewriting.
    2. Document retrieval based on the (possibly rewritten) query.
    3. Answer generation using a language model and the retrieved context.

    It must return a dictionary with 'answer', 'context_chunks', 'rewrite_strategy', and 'retrieval_score'.
    """
    if verbose:
        print(f"Executing RAG pipeline for query: {query}")

    # --- Placeholder Logic ---
    # In a real scenario, this would involve calling your LLM, vector DB, etc.

    # Simulate retrieval
    retrieved_contexts = [
        "Retrieval Augmented Generation (RAG) is an AI framework for improving the relevancy and factual accuracy of generative AI models.",
        "Attention mechanisms in transformers allow the model to weigh the importance of different parts of the input sequence.",
        "Dense retrieval models represent documents and queries as dense vectors in a continuous space, allowing for efficient similarity search."
    ]

    # Simulate answer generation based on a simple keyword match for demonstration
    answer = ""
    if "retrieval augmented generation" in query.lower():
        answer = "Retrieval Augmented Generation (RAG) is an AI framework that enhances generative AI models by incorporating factual information from external sources."
    elif "attention" in query.lower() and "transformers" in query.lower():
        answer = "Attention in transformers allows the model to dynamically focus on relevant parts of the input sequence, improving contextual understanding."
    elif "dense retrieval" in query.lower():
        answer = "Dense retrieval offers advantages like semantic understanding and better handling of lexical gaps compared to sparse retrieval."
    else:
        answer = f"I am a placeholder RAG pipeline and cannot fully answer '{query}'."

    pipeline_result = {
        "answer": answer,
        "context_chunks": retrieved_contexts,
        "rewrite_strategy": "original", # Or 'rewritten' if applicable
        "retrieval_score": 0.8 # Placeholder score
    }

    return pipeline_result

print("✅ Placeholder `full_rag_pipeline` function defined.")

✅ Placeholder `full_rag_pipeline` function defined.


In [5]:
# Create a test set for evaluation
test_cases = [
    {"query": "What is retrieval augmented generation?", "ground_truth": None},
    {"query": "How does attention work in transformers?", "ground_truth": None},
    {"query": "What are the benefits of dense retrieval?", "ground_truth": None},
]

evaluation_results = []

for case in test_cases:
    print(f"\n🧪 Testing: {case['query'][:50]}...")
    pipeline_result = full_rag_pipeline(case["query"], verbose=False)

    scores = evaluate_with_ragas(
        query=case["query"],
        answer=pipeline_result["answer"],
        contexts=pipeline_result["context_chunks"],
        ground_truth=case.get("ground_truth")
    )

    row = {
        "query": case["query"],
        "strategy": pipeline_result["rewrite_strategy"],
        "retrieval_score": pipeline_result["retrieval_score"],
        **scores
    }
    evaluation_results.append(row)
    print(f"   Faithfulness: {scores['faithfulness']:.3f} | Relevancy: {scores['answer_relevancy']:.3f}")

import pandas as pd
df = pd.DataFrame(evaluation_results)
print("\n📊 Evaluation Summary:")
print(df[["query", "strategy", "faithfulness", "answer_relevancy"]].to_string(index=False))
print(f"\nMean Faithfulness: {df['faithfulness'].mean():.3f}")
print(f"Mean Relevancy: {df['answer_relevancy'].mean():.3f}")


🧪 Testing: What is retrieval augmented generation?...
⚠️ RAGAS failed (No module named 'ragas'), using heuristic fallback
   Faithfulness: 0.579 | Relevancy: 1.000

🧪 Testing: How does attention work in transformers?...
⚠️ RAGAS failed (No module named 'ragas'), using heuristic fallback
   Faithfulness: 0.556 | Relevancy: 0.667

🧪 Testing: What are the benefits of dense retrieval?...
⚠️ RAGAS failed (No module named 'ragas'), using heuristic fallback
   Faithfulness: 0.294 | Relevancy: 0.571

📊 Evaluation Summary:
                                    query strategy  faithfulness  answer_relevancy
  What is retrieval augmented generation? original      0.578947          1.000000
 How does attention work in transformers? original      0.555556          0.666667
What are the benefits of dense retrieval? original      0.294118          0.571429

Mean Faithfulness: 0.476
Mean Relevancy: 0.746


In [9]:
import sqlite3
import json
from datetime import datetime
from pathlib import Path

DB_PATH = "/content/drive/MyDrive/SelfImprovingRAG/feedback.db"

def init_feedback_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""
        CREATE TABLE IF NOT EXISTS feedback_records (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            query TEXT, answer TEXT,
            user_rating INTEGER,
            faithfulness_score REAL, relevancy_score REAL,
            strategy_used TEXT,
            bm25_weight REAL, vector_weight REAL,
            retrieval_score REAL, overall_reward REAL,
            session_id TEXT,
            created_at TEXT DEFAULT CURRENT_TIMESTAMP
        )""")
    c.execute("""
        CREATE TABLE IF NOT EXISTS strategy_weights (
            strategy_name TEXT PRIMARY KEY,
            weight REAL DEFAULT 0.25,
            sample_count INTEGER DEFAULT 0,
            avg_reward REAL DEFAULT 0.5
        )""")
    for s in ["sub_query_decomposition","step_back_prompting","hyde","multi_query"]:
        c.execute("INSERT OR IGNORE INTO strategy_weights (strategy_name) VALUES (?)", (s,))
    conn.commit()
    conn.close()
    print("✅ Feedback database initialized")

def save_feedback(query, answer, strategy, faithfulness, relevancy,
                  user_rating=None, session_id=""):
    reward = (faithfulness + relevancy) / 2
    if user_rating == 1: reward = min(reward + 0.3, 1.0)
    elif user_rating == -1: reward = max(reward - 0.4, 0.0)

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""
        INSERT INTO feedback_records
        (query, answer, user_rating, faithfulness_score, relevancy_score,
         strategy_used, bm25_weight, vector_weight, retrieval_score, overall_reward, session_id)
        VALUES (?,?,?,?,?,?,?,?,?,?,?)""",
        (query, answer, user_rating, faithfulness, relevancy,
         strategy, 0.3, 0.7, faithfulness, reward, session_id))
    conn.commit()

    # Update strategy weights (EMA)
    c.execute("SELECT avg_reward, sample_count FROM strategy_weights WHERE strategy_name=?", (strategy,))
    row = c.fetchone()
    if row:
        old_avg, count = row
        new_avg = 0.9 * old_avg + 0.1 * reward  # EMA with alpha=0.1
        c.execute("""UPDATE strategy_weights SET avg_reward=?, sample_count=?+1
                     WHERE strategy_name=?""", (new_avg, count, strategy))
    conn.commit()
    conn.close()
    return reward

def get_strategy_weights():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT strategy_name, avg_reward FROM strategy_weights")
    rows = c.fetchall()
    conn.close()
    weights = {s: r for s, r in rows}
    total = sum(weights.values()) or 1
    return {s: r/total for s, r in weights.items()}

init_feedback_db()
print("✅ Feedback store ready")

✅ Feedback database initialized
✅ Feedback store ready


In [10]:
import random

print("🔄 Simulating feedback loop...\n")

# Save all evaluation results as feedback
for i, (case, result, scores) in enumerate(zip(test_cases, [full_rag_pipeline(c["query"], verbose=False) for c in test_cases], evaluation_results)):
    # Simulate user ratings (in real app, user clicks thumbs up/down)
    simulated_rating = 1 if scores["faithfulness"] > 0.6 else -1

    reward = save_feedback(
        query=case["query"],
        answer="[see pipeline]",
        strategy=result["rewrite_strategy"],
        faithfulness=scores["faithfulness"],
        relevancy=scores["answer_relevancy"],
        user_rating=simulated_rating,
        session_id=f"test_session_{i}"
    )
    print(f"  Saved feedback: strategy={result['rewrite_strategy']}, reward={reward:.3f}, user_rating={simulated_rating}")

# Show updated weights
print("\n📊 Current Strategy Weights (after learning):")
weights = get_strategy_weights()
for strategy, weight in sorted(weights.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(weight * 40)
    print(f"  {strategy:30s} {bar} {weight:.4f}")

🔄 Simulating feedback loop...

  Saved feedback: strategy=original, reward=0.389, user_rating=-1
  Saved feedback: strategy=original, reward=0.211, user_rating=-1
  Saved feedback: strategy=original, reward=0.033, user_rating=-1

📊 Current Strategy Weights (after learning):
  sub_query_decomposition        ██████████ 0.2500
  step_back_prompting            ██████████ 0.2500
  hyde                           ██████████ 0.2500
  multi_query                    ██████████ 0.2500
